# Deepfake Detection - Complete Training Pipeline

This notebook contains the complete pipeline for training and testing a CNN-LSTM model for deepfake detection.

## 1. Configuration

In [ ]:
# Image settings
IMG_SIZE = 380  # EfficientNet-B4 input size
NUM_FRAMES = 32

# Dataset paths
ROOT_DIR = r"path/to/video_folder"  # Thư mục chứa các file video

# CSV file paths (each file format: Unnamed: 0, File Path, Label, Frame Count, Width, Height, Codec)
# Where Label column contains: "REAL" (or 0) for original videos, "FAKE" (or 1) for deepfake videos
ORIGINAL_CSV = r"\FaceForensics++_C23\csv\original.csv"          # 1000 video gốc
DEEPFAKES_CSV = r"\FaceForensics++_C23\csv\Deepfakes.csv"        # 1000 video Deepfakes
FACE2FACE_CSV = r"\FaceForensics++_C23\csv\Face2Face.csv"        # 1000 video Face2Face
FACESHIFTER_CSV = r"\FaceForensics++_C23\csv\FaceShifter.csv"    # 1000 video FaceShifter
FACESWAP_CSV = r"\FaceForensics++_C23\csv\FaceSwap.csv"          # 1000 video FaceSwap
NEURALTEXTURES_CSV = r"\FaceForensics++_C23\csv\NeuralTextures.csv"  # 1000 video NeuralTextures

# Data split ratios (each method)
TRAIN_RATIO = 0.72   # 70% training
VAL_RATIO = 0.14    # 15% validation
TEST_RATIO = 0.14   # 15% testing
RANDOM_SEED = 42

# DataLoader settings
BATCH_SIZE = 8
NUM_WORKERS = 4
PIN_MEMORY = True

# Model settings
MODEL_HIDDEN_SIZE = 256
MODEL_NUM_LAYERS = 1
MODEL_DROPOUT = 0.3

# Training settings
EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# Checkpoint
SAVE_PATH = "best_efficientb4.pth"
DEVICE = "cuda"

# External Test Dataset Configuration
TEST_TXT_FILE = r"/celeb-df-v2/List_of_testing_videos.txt"
TEST_ROOT_DIR = r"/celeb-df-v2"
TEST_USE_FACE_DETECTION = True

## 2. Imports

In [ ]:
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from facenet_pytorch import MTCNN
from tqdm import tqdm
import os
import argparse
import warnings
warnings.filterwarnings('ignore')

## 3. Transform - Data Augmentation

In [ ]:
def get_train_transform():
    """Get training augmentation transforms"""
    return A.Compose(
        [
            A.Resize(IMG_SIZE, IMG_SIZE),
            # flip
            A.HorizontalFlip(p=0.5),
            # xoay
            A.Rotate(limit=20, p=0.5),
            # mờ
            A.GaussianBlur(blur_limit=(3, 7), p=0.3),
            # nhiễu
            A.GaussNoise(std_range=(0.02, 0.1), p=0.3),
            # cắt
            A.RandomResizedCrop(
                size=(IMG_SIZE, IMG_SIZE),
                scale=(0.7, 1.0),
                ratio=(0.9, 1.1),
                p=0.5
            ),
            # normalize
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
            ToTensorV2()
        ],
        additional_targets={f"image{i}": "image" for i in range(1, NUM_FRAMES)}
    )


def get_val_transform():
    """Get validation transforms (no augmentation)"""
    return A.Compose(
        [
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
            ToTensorV2()
        ],
        additional_targets={f"image{i}": "image" for i in range(1, NUM_FRAMES)}
    )

## 4. Utils - Utility Functions

In [ ]:
def apply_transform_to_video(frames, transform):
    """
    Áp dụng cùng một transform cho toàn bộ frames của video
    
    Args:
        frames: numpy array (T, H, W, C) - ví dụ (16, 224, 224, 3)
        transform: albumentations transform
        
    Returns:
        tensor: (T, C, H, W)
    """
    data = {"image": frames[0]}

    for i in range(1, len(frames)):
        data[f"image{i}"] = frames[i]

    augmented = transform(**data)

    transformed_frames = [augmented["image"]]
    for i in range(1, len(frames)):
        img = augmented[f"image{i}"]
        # Chuyển đổi numpy array thành tensor nếu cần
        if isinstance(img, np.ndarray):
            img = torch.from_numpy(img).permute(2, 0, 1).float()
        transformed_frames.append(img)

    video_tensor = torch.stack(transformed_frames)  # (T, C, H, W)

    return video_tensor


def video_collate_fn(batch):
    """
    Custom collate function for video batch
    
    Args:
        batch: list[(video_tensor, label)]
            video_tensor shape: (T, C, H, W)
            
    Returns:
        tensors: (videos, labels)
            videos shape: (B, T, C, H, W)
            labels shape: (B,)
    """
    videos = []
    labels = []

    for item in batch:
        if item is None:
            continue

        video, label = item

        # đảm bảo đủ frames theo config
        if video.shape[0] != NUM_FRAMES:
            continue

        videos.append(video)
        labels.append(label)

    if len(videos) == 0:
        return None

    videos = torch.stack(videos)   # (B, T, C, H, W)
    labels = torch.stack(labels)   # (B,)

    return videos, labels

## 5. Dataset

In [ ]:
class DeepfakeVideoDataset(Dataset):
    """Dataset for loading video frames and labels"""
    
    def __init__(
        self,
        video_paths,
        labels,
        root_dir="",
        transform=None,
        num_frames=NUM_FRAMES,
        device="cuda",
        use_face_detection=True
    ):
        """
        Args:
            video_paths: List of video file paths
            labels: List of labels (0 for original, 1 for fake)
            root_dir: Root directory for video paths
            transform: Transform to apply to frames
            num_frames: Number of frames to extract from each video
            device: Device to use (cuda or cpu)
            use_face_detection: Whether to use MTCNN for face detection
        """
        self.video_paths = video_paths
        self.labels = labels
        self.root_dir = root_dir
        self.transform = transform
        self.num_frames = num_frames
        self.device = device
        self.use_face_detection = use_face_detection
        
        # Initialize MTCNN detector if face detection is enabled
        if self.use_face_detection:
            self.face_detector = MTCNN(device=device, keep_all=False)

    def __len__(self):
        return len(self.video_paths)

    def extract_frames(self, video_path):
        """Extract evenly spaced frames from video"""
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            cap.release()
            return []

        indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)

        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                
                # Apply face detection if enabled
                if self.use_face_detection:
                    frame = self._detect_and_crop_face(frame)
                    if frame is None:
                        continue
                
                frames.append(frame)

        cap.release()
        return frames
    
    def _detect_and_crop_face(self, frame):
        """Detect face using MTCNN and crop it"""
        try:
            # Detect faces in the frame
            boxes, probs, landmarks = self.face_detector.detect(frame, landmarks=True)
            
            if boxes is None or len(boxes) == 0:
                return None
            
            # Get the largest face (highest confidence)
            if len(probs) > 0:
                best_idx = probs.argmax()
                box = boxes[best_idx].cpu().numpy() if torch.is_tensor(boxes[best_idx]) else boxes[best_idx]
            else:
                box = boxes[0].cpu().numpy() if torch.is_tensor(boxes[0]) else boxes[0]
            
            # Extract bounding box (x1, y1, x2, y2)
            x1, y1, x2, y2 = box.astype(int)
            
            # Add padding to the bounding box
            padding = 20
            x1 = max(0, x1 - padding)
            y1 = max(0, y1 - padding)
            x2 = min(frame.shape[1], x2 + padding)
            y2 = min(frame.shape[0], y2 + padding)
            
            # Crop face region
            face_crop = frame[y1:y2, x1:x2]
            
            # Check if crop is valid
            if face_crop.size == 0:
                return None
            
            # Resize to IMG_SIZE
            face_crop = cv2.resize(face_crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            
            return face_crop
        except Exception as e:
            # Log error and return None if detection fails
            print(f"Face detection error: {e}")
            return None

    def __getitem__(self, idx):
        video_path = str(self.video_paths[idx])  # Convert to string to handle numpy int64
        
        # Ghép root_dir nếu cần
        if self.root_dir:
            video_path = os.path.join(self.root_dir, video_path)
        
        label = self.labels[idx]
        
        frames = self.extract_frames(video_path)
        
        if len(frames) == 0:
            # fallback nếu video lỗi
            frames = np.zeros((self.num_frames, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        
        if self.transform:
            frames = apply_transform_to_video(frames, self.transform)
        
        return frames, torch.tensor(label, dtype=torch.long)

## 6. DataLoader

In [ ]:
def _convert_label_to_numeric(label):
    """Convert label to numeric (REAL → 0, FAKE → 1)"""
    if isinstance(label, str):
        if label == 'REAL':
            return 0
        elif label == 'FAKE':
            return 1
    # Handle numeric labels
    return int(label)


def create_dataloaders(
    root_dir,
    original_csv,
    deepfakes_csv,
    face2face_csv,
    faceshifter_csv,
    faceswap_csv,
    neuraltextures_csv,
    train_ratio=0.72,
    val_ratio=0.14,
    test_ratio=0.14,
    random_seed=42,
    use_face_detection=True
):
    """
    Create train, validation, and test dataloaders from multiple CSV files.
    
    Each CSV file has columns: File Path, Label, Frame Count, Width, Height, Codec, File Size(MB)
    Where Label is either: "REAL"/"FAKE" or 0/1
    
    Args:
        root_dir: Root directory for videos
        original_csv: Path to CSV with original/real video paths (contains Label column)
        deepfakes_csv: Path to CSV with Deepfakes video paths (contains Label column)
        face2face_csv: Path to CSV with Face2Face video paths (contains Label column)
        faceshifter_csv: Path to CSV with FaceShifter video paths (contains Label column)
        faceswap_csv: Path to CSV with FaceSwap video paths (contains Label column)
        neuraltextures_csv: Path to CSV with NeuralTextures video paths (contains Label column)
        train_ratio: Ratio for training set (default: 0.7)
        val_ratio: Ratio for validation set (default: 0.15)
        test_ratio: Ratio for testing set (default: 0.15)
        random_seed: Random seed for reproducibility
        use_face_detection: Whether to use MTCNN for face detection
        
    Returns:
        train_loader, val_loader, test_loader
    """
    
    # Set random seed for reproducibility
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    
    # Dictionary of CSV files for fake methods
    fake_csv_files = {
        "Deepfakes": deepfakes_csv,
        "Face2Face": face2face_csv,
        "FaceShifter": faceshifter_csv,
        "FaceSwap": faceswap_csv,
        "NeuralTextures": neuraltextures_csv
    }
    
    # Helper function to load and split data from CSV
    def load_and_split_csv(csv_path, train_ratio, val_ratio):
        """Load CSV and split data into train/val/test"""
        df = pd.read_csv(csv_path)
        
        if 'File Path' not in df.columns:
            raise ValueError(f"CSV must contain 'File Path' column. Found columns: {df.columns.tolist()}")
        if 'Label' not in df.columns:
            raise ValueError(f"CSV must contain 'Label' column. Found columns: {df.columns.tolist()}")
        
        video_paths = df['File Path'].values
        labels = df['Label'].values
        
        # Convert string labels to numeric
        labels = np.array([_convert_label_to_numeric(label) for label in labels])
        
        # Calculate split sizes
        train_size = int(len(video_paths) * train_ratio)
        val_size = int(len(video_paths) * val_ratio)
        
        # Split without randomization
        return (
            video_paths[:train_size], labels[:train_size],
            video_paths[train_size:train_size + val_size], labels[train_size:train_size + val_size],
            video_paths[train_size + val_size:], labels[train_size + val_size:]
        )
    
    # Load original videos using the helper function
    original_train_paths, original_train_labels, \
    original_val_paths, original_val_labels, \
    original_test_paths, original_test_labels = load_and_split_csv(original_csv, train_ratio, val_ratio)
    
    # Collect all videos and labels
    all_train_videos = list(original_train_paths)
    all_train_labels = list(original_train_labels)
    
    all_val_videos = list(original_val_paths)
    all_val_labels = list(original_val_labels)
    
    all_test_videos = list(original_test_paths)
    all_test_labels = list(original_test_labels)
    
    # Load fake videos from each method
    for method_name, csv_path in fake_csv_files.items():
        fake_train_paths, fake_train_labels, \
        fake_val_paths, fake_val_labels, \
        fake_test_paths, fake_test_labels = load_and_split_csv(csv_path, train_ratio, val_ratio)
        
        # Add to lists
        all_train_videos.extend(list(fake_train_paths))
        all_train_labels.extend(list(fake_train_labels))
        
        all_val_videos.extend(list(fake_val_paths))
        all_val_labels.extend(list(fake_val_labels))
        
        all_test_videos.extend(list(fake_test_paths))
        all_test_labels.extend(list(fake_test_labels))
    
    # Get transforms
    train_transform = get_train_transform()
    val_transform = get_val_transform()
    
    # Create train dataset
    train_dataset = DeepfakeVideoDataset(
        video_paths=all_train_videos,
        labels=all_train_labels,
        root_dir=root_dir,
        transform=train_transform,
        num_frames=NUM_FRAMES,
        device="cuda",
        use_face_detection=use_face_detection
    )
    
    # Create validation dataset
    val_dataset = DeepfakeVideoDataset(
        video_paths=all_val_videos,
        labels=all_val_labels,
        root_dir=root_dir,
        transform=val_transform,
        num_frames=NUM_FRAMES,
        device="cuda",
        use_face_detection=use_face_detection
    )
    
    # Create test dataset
    test_dataset = DeepfakeVideoDataset(
        video_paths=all_test_videos,
        labels=all_test_labels,
        root_dir=root_dir,
        transform=val_transform,
        num_frames=NUM_FRAMES,
        device="cuda",
        use_face_detection=use_face_detection
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=video_collate_fn
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=video_collate_fn
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=video_collate_fn
    )
    
    return train_loader, val_loader, test_loader

## 7. Model - CNN-LSTM Architecture

In [ ]:
class CNN_LSTM(nn.Module):
    """CNN-LSTM model combining EfficientNet-B0 with LSTM"""
    
    def __init__(self, hidden_size=256, num_layers=1, dropout=0.3):
        """
        Args:
            hidden_size: Size of LSTM hidden state
            num_layers: Number of LSTM layers
            dropout: Dropout rate
        """
        super().__init__()

        # EfficientNet-B0 backbone
        backbone = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
        )

        # bỏ classifier cuối
        self.cnn = backbone.features
        self.pool = backbone.avgpool

        self.feature_dim = 1280  # output EfficientNet-B0

        # LSTM
        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: (B, T, C, H, W) - batch of video sequences
            
        Returns:
            logits: (B,) - prediction logits for each video
        """
        B, T, C, H, W = x.shape

        # gộp batch và time
        x = x.view(B * T, C, H, W)

        # CNN
        features = self.cnn(x)          # (B*T, 1792, H', W')
        features = self.pool(features)  # (B*T, 1792, 1, 1)
        features = features.view(B * T, -1)

        # reshape lại theo sequence
        features = features.view(B, T, -1)  # (B, T, 1792)

        lstm_out, _ = self.lstm(features)

        # lấy frame cuối
        last_out = lstm_out[:, -1, :]

        out = self.fc(last_out)
        return out.squeeze(1)


def create_model(device="cuda", hidden_size=256, num_layers=1, dropout=0.3, checkpoint_path=None):
    """
    Create and initialize model
    
    Args:
        device: Device to use (cuda or cpu)
        hidden_size: LSTM hidden size
        num_layers: Number of LSTM layers
        dropout: Dropout rate
        checkpoint_path: Path to checkpoint file to load weights from
        
    Returns:
        model: Initialized model
    """
    model = CNN_LSTM(
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model created on {device}")
    print(f"Total parameters: {total_params:,}")
    
    # Use SAVE_PATH from config if checkpoint_path not specified
    if checkpoint_path is None:
        checkpoint_path = SAVE_PATH
    
    # Load pretrained weights if checkpoint exists
    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"\nLoading checkpoint from: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        
        # Handle both full model state and state_dict
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"Loaded model weights")
            if 'epoch' in checkpoint:
                print(f"Checkpoint from epoch: {checkpoint['epoch']}")
        else:
            # Assume it's directly the state_dict
            model.load_state_dict(checkpoint)
            print(f"Loaded model weights")
    else:
        if checkpoint_path:
            print(f"\nCheckpoint not found at: {checkpoint_path}")
            print("Training from scratch with ImageNet-pretrained EfficientNet-B0")

    return model

## 8. Metrics

In [ ]:
def compute_metrics(outputs, labels):
    """
    Compute accuracy, AUC, precision, recall, and F1 score metrics
    
    Args:
        outputs: Model output logits (B,)
        labels: Ground truth labels (B,)
        
    Returns:
        acc: Accuracy score
        auc: ROC-AUC score
        precision: Precision (TP / (TP + FP))
        recall: Recall (TP / (TP + FN))
        f1: F1 score (harmonic mean of precision and recall)
    """
    probs = torch.sigmoid(outputs).detach().cpu().numpy()
    labels = labels.detach().cpu().numpy()

    preds = (probs > 0.5).astype(int)

    acc = (preds == labels).mean()

    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = 0.5

    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    return acc, auc, precision, recall, f1

## 9. Training

In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting"""
    
    def __init__(self, patience=5):
        """
        Args:
            patience: Number of epochs to wait before stopping
        """
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.early_stop = False

    def step(self, val_loss):
        """
        Check if validation loss improved
        
        Args:
            val_loss: Validation loss
            
        Returns:
            should_save: Whether to save the model
        """
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            return True  # save model
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False


def train_one_epoch(model, dataloader, optimizer, criterion, device):
    """
    Train for one epoch
    
    Args:
        model: Model to train
        dataloader: Training dataloader
        optimizer: Optimizer
        criterion: Loss function
        device: Device to use
        
    Returns:
        avg_loss: Average loss
        acc: Accuracy
        auc: AUC score
        precision: Precision score
        recall: Recall score
        f1: F1 score
    """
    model.train()

    total_loss = 0
    all_outputs = []
    all_labels = []

    pbar = tqdm(dataloader, desc="Training")

    for batch in pbar:
        if batch is None:
            continue

        videos, labels = batch
        videos = videos.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()

        outputs = model(videos)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        all_outputs.append(outputs)
        all_labels.append(labels)

        pbar.set_postfix({"loss": loss.item()})

    all_outputs = torch.cat(all_outputs)
    all_labels = torch.cat(all_labels)

    avg_loss = total_loss / len(all_labels)
    acc, auc, precision, recall, f1 = compute_metrics(all_outputs, all_labels)

    return avg_loss, acc, auc, precision, recall, f1


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    """
    Validate the model
    
    Args:
        model: Model to validate
        dataloader: Validation dataloader
        criterion: Loss function
        device: Device to use
        
    Returns:
        avg_loss: Average loss
        acc: Accuracy
        auc: AUC score
        precision: Precision score
        recall: Recall score
        f1: F1 score
    """
    model.eval()

    total_loss = 0
    all_outputs = []
    all_labels = []

    pbar = tqdm(dataloader, desc="Validation")

    for batch in pbar:
        if batch is None:
            continue

        videos, labels = batch
        videos = videos.to(device)
        labels = labels.float().to(device)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)

        all_outputs.append(outputs)
        all_labels.append(labels)

    all_outputs = torch.cat(all_outputs)
    all_labels = torch.cat(all_labels)

    avg_loss = total_loss / len(all_labels)
    acc, auc, precision, recall, f1 = compute_metrics(all_outputs, all_labels)

    return avg_loss, acc, auc, precision, recall, f1


def train_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    optimizer,
    criterion,
    device,
    epochs=30,
    patience=5,
    save_path="best_efficientb0.pth"
):
    """
    Train the model with test evaluation
    
    Args:
        model: Model to train
        train_loader: Training dataloader
        val_loader: Validation dataloader
        test_loader: Test dataloader
        optimizer: Optimizer
        criterion: Loss function
        device: Device to use
        epochs: Number of epochs to train
        patience: Early stopping patience
        save_path: Path to save the best model
    
    Prints:
        Training, validation, and test metrics for each epoch
    """
    early_stopping = EarlyStopping(patience=patience)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        train_loss, train_acc, train_auc, train_precision, train_recall, train_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        val_loss, val_acc, val_auc, val_precision, val_recall, val_f1 = validate(
            model, val_loader, criterion, device
        )
        
        # Test evaluation
        test_loss, test_acc, test_auc, test_precision, test_recall, test_f1 = validate(
            model, test_loader, criterion, device
        )

        print(
            f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f} | Precision: {train_precision:.4f} | Recall: {train_recall:.4f} | F1: {train_f1:.4f}"
        )
        print(
            f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f} | Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f}"
        )
        print(
            f"Test  Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | AUC: {test_auc:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f} | F1: {test_f1:.4f}"
        )

        # early stopping check
        should_save = early_stopping.step(val_loss)

        if should_save:
            print("Saving best model...")
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                    "test_loss": test_loss,
                    "test_acc": test_acc,
                    "test_auc": test_auc,
                    "epoch": epoch,
                },
                save_path
            )

        if early_stopping.early_stop:
            print("Early stopping triggered")
            break

    print(f"\nTraining completed. Best model saved to {save_path}")

## 10. Test - External Dataset Evaluation

In [ ]:
class TestVideoDataset(Dataset):
    """Dataset for loading external test videos from txt file"""
    
    def __init__(self, txt_file, root_dir, transform=None, num_frames=NUM_FRAMES, 
                 device="cuda", use_face_detection=TEST_USE_FACE_DETECTION):
        """
        Args:
            txt_file: Text file containing labels and video paths
                     Format: label path_to_video (e.g., "0 path/to/video.mp4")
            root_dir: Root directory containing the videos
            transform: Transform to apply to frames
            num_frames: Number of frames to extract from each video
            device: Device to use (cuda or cpu)
            use_face_detection: Whether to use MTCNN for face detection
        """
        self.root_dir = root_dir
        self.transform = transform
        self.num_frames = num_frames
        self.device = device
        self.use_face_detection = use_face_detection

        self.video_paths = []
        self.labels = []

        # Read txt file
        with open(txt_file, "r") as f:
            lines = f.readlines()

        for line in lines:
            line = line.strip()
            if line == "":
                continue

            parts = line.split(" ", 1)
            if len(parts) != 2:
                continue
                
            label, path = parts
            video_path = os.path.join(root_dir, path)
            
            # Check if video file exists
            if os.path.exists(video_path):
                self.video_paths.append(video_path)
                # Swap labels: 0 (fake/synthesis) -> 1, 1 (real) -> 0
                flipped_label = 1 - int(label)
                self.labels.append(flipped_label)
            else:
                print(f"Warning: Video file not found: {video_path}")

        print(f"Loaded {len(self.video_paths)} videos from {txt_file}")
        
        # Initialize MTCNN detector if face detection is enabled
        if self.use_face_detection:
            self.face_detector = MTCNN(device=device, keep_all=False)
        else:
            self.face_detector = None

    def __len__(self):
        return len(self.video_paths)

    def extract_frames(self, video_path):
        """Extract evenly spaced frames from video"""
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            cap.release()
            return []

        indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)

        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                continue

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Apply face detection if enabled
            if self.use_face_detection and self.face_detector:
                frame = self._detect_and_crop_face(frame)
                if frame is None:
                    continue
            
            frames.append(frame)

        cap.release()

        # Pad with last frame if insufficient frames
        while len(frames) < self.num_frames:
            if len(frames) > 0:
                frames.append(frames[-1])
            else:
                # Create blank frame if no frames available
                frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

        return frames

    def _detect_and_crop_face(self, frame):
        """Detect face using MTCNN and crop it"""
        try:
            # Detect faces in the frame
            boxes, probs, landmarks = self.face_detector.detect(frame, landmarks=True)
            
            if boxes is None or len(boxes) == 0:
                return None
            
            # Get the largest face (highest confidence)
            if len(probs) > 0:
                best_idx = probs.argmax()
                box = boxes[best_idx].cpu().numpy() if torch.is_tensor(boxes[best_idx]) else boxes[best_idx]
            else:
                box = boxes[0].cpu().numpy() if torch.is_tensor(boxes[0]) else boxes[0]
            
            # Extract bounding box (x1, y1, x2, y2)
            x1, y1, x2, y2 = box.astype(int)
            
            # Add padding to the bounding box
            padding = 20
            x1 = max(0, x1 - padding)
            y1 = max(0, y1 - padding)
            x2 = min(frame.shape[1], x2 + padding)
            y2 = min(frame.shape[0], y2 + padding)
            
            # Crop face region
            face_crop = frame[y1:y2, x1:x2]
            
            # Check if crop is valid
            if face_crop.size == 0:
                return None
            
            # Resize to IMG_SIZE
            face_crop = cv2.resize(face_crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            
            return face_crop
        except Exception as e:
            # Log error and return None if detection fails
            print(f"Face detection error: {e}")
            return None

    def apply_transform_to_video(self, frames):
        """Apply transform to video frames"""
        if self.transform:
            frames = apply_transform_to_video(frames, self.transform)
        else:
            # Convert to tensor manually if no transform
            frames = torch.from_numpy(np.array(frames)).permute(0, 3, 1, 2).float()
        return frames

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        frames = self.extract_frames(video_path)
        
        if len(frames) == 0:
            # Fallback if video fails to load
            frames = [np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) for _ in range(self.num_frames)]
        
        frames = self.apply_transform_to_video(frames)

        # Output: (T, C, H, W)
        return frames, torch.tensor(label, dtype=torch.long)


def evaluate_model(model, test_loader, device):
    """
    Evaluate model on test dataset and compute metrics
    
    Args:
        model: Model to evaluate
        test_loader: DataLoader for test data
        device: Device to use
        
    Returns:
        Dictionary with computed metrics
    """
    model.eval()
    
    all_outputs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating batches"):
            if batch is None:
                continue
                
            videos, labels = batch
            videos = videos.to(device)
            labels = labels.to(device)
            
            # Forward pass
            logits = model(videos)
            
            all_outputs.append(logits)
            all_labels.append(labels)
    
    # Concatenate all outputs and labels
    all_outputs = torch.cat(all_outputs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    
    # Compute metrics
    acc, auc, precision, recall, f1 = compute_metrics(all_outputs, all_labels)
    
    return {
        'accuracy': acc,
        'auc': auc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'num_samples': len(all_labels)
    }

## 11. Main Execution - Training Pipeline

In [ ]:
# Example usage: Initialize and train the model
if __name__ == "__main__":
    # Create dataloaders
    print("Creating dataloaders...")
    train_loader, val_loader, test_loader = create_dataloaders(
        root_dir=ROOT_DIR,
        original_csv=ORIGINAL_CSV,
        deepfakes_csv=DEEPFAKES_CSV,
        face2face_csv=FACE2FACE_CSV,
        faceshifter_csv=FACESHIFTER_CSV,
        faceswap_csv=FACESWAP_CSV,
        neuraltextures_csv=NEURALTEXTURES_CSV,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        random_seed=RANDOM_SEED,
        use_face_detection=True
    )
    
    # Create model
    print("\nCreating model...")
    model = create_model(
        device=DEVICE,
        hidden_size=MODEL_HIDDEN_SIZE,
        num_layers=MODEL_NUM_LAYERS,
        dropout=MODEL_DROPOUT
    )
    
    # Setup optimizer and loss function
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    
    # Train the model
    print("\nStarting training...")
    train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
        epochs=EPOCHS,
        patience=PATIENCE,
        save_path=SAVE_PATH
    )

## 12. External Test - Evaluate on External Dataset

In [ ]:
def test_external_dataset(txt_file, root_dir, checkpoint_path=None):
    """
    Test model on external dataset
    
    Args:
        txt_file: Path to txt file with test data
        root_dir: Root directory for test videos
        checkpoint_path: Path to model checkpoint
    """
    # Create test dataset
    print("Loading test dataset...")
    test_dataset = TestVideoDataset(
        txt_file=txt_file,
        root_dir=root_dir,
        transform=get_val_transform(),
        num_frames=NUM_FRAMES,
        device=DEVICE,
        use_face_detection=TEST_USE_FACE_DETECTION
    )
    
    # Create dataloader
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=video_collate_fn
    )
    
    # Create model
    print("Loading model...")
    model = create_model(
        device=DEVICE,
        hidden_size=MODEL_HIDDEN_SIZE,
        num_layers=MODEL_NUM_LAYERS,
        dropout=MODEL_DROPOUT,
        checkpoint_path=checkpoint_path or SAVE_PATH
    )
    
    # Evaluate
    print("\nEvaluating on external dataset...")
    metrics = evaluate_model(model, test_loader, DEVICE)
    
    print("\n" + "="*50)
    print("EXTERNAL TEST RESULTS")
    print("="*50)
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"AUC:       {metrics['auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1-Score:  {metrics['f1']:.4f}")
    print(f"Num Samples: {metrics['num_samples']}")
    print("="*50)
    
    return metrics


# Example: Test on external dataset
# metrics = test_external_dataset(
#     txt_file=TEST_TXT_FILE,
#     root_dir=TEST_ROOT_DIR,
#     checkpoint_path=SAVE_PATH
# )